In [ ]:
!pip -q install sentence-transformers seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
sentences = [
    # Cricket
    "The batsman scored a century in the final match.",
    "The bowler delivered a fast yorker that knocked over the stumps.",
    "The team chased the target with two overs remaining.",

    # Cooking
    "The chef simmered the tomato sauce for thirty minutes.",
    "She baked a chocolate cake with fresh cream frosting.",
    "The recipe needs garlic, onions, and olive oil.",

    # Cybersecurity
    "The security analyst detected malware in the email attachment.",
    "Strong passwords and multi-factor authentication improve account security.",
    "The firewall blocked suspicious traffic from an unknown IP address.",
    "The company responded quickly to the ransomware attack."
]

topics = [
    "Cricket", "Cricket", "Cricket",
    "Cooking", "Cooking", "Cooking",
    "Cybersecurity", "Cybersecurity", "Cybersecurity", "Cybersecurity"
]

df = pd.DataFrame({
    "sentence_id": range(1, 11),
    "topic": topics,
    "sentence": sentences
})

df

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(sentences)

print("Embeddings shape:", embeddings.shape)

In [ ]:
similarity_matrix = cosine_similarity(embeddings, embeddings)

sim_df = pd.DataFrame(
    similarity_matrix,
    index=[f"S{i}" for i in range(1, 11)],
    columns=[f"S{i}" for i in range(1, 11)]
)

sim_df.round(3)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(sim_df, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("10×10 Cosine Similarity Matrix")
plt.xlabel("Sentences")
plt.ylabel("Sentences")
plt.tight_layout()
plt.show()

In [ ]:
query_sentence = "The bowler took three wickets in one over"
query_embedding = model.encode([query_sentence])

query_similarities = cosine_similarity(query_embedding, embeddings)[0]

results = pd.DataFrame({
    "sentence": sentences,
    "topic": topics,
    "similarity_score": query_similarities
}).sort_values(by="similarity_score", ascending=False).reset_index(drop=True)

top2 = results.head(2)
top2

In [ ]:
print("Query sentence:", query_sentence)
print("\nTop 2 most similar sentences:\n")

for i, row in top2.iterrows():
    print(f"{i+1}. [{row['topic']}] {row['sentence']}")
    print(f"   Similarity score: {row['similarity_score']:.4f}")